# Lab 02 solution, notebook 2: a third lookup, and an empty result

The worked answers to the two challenge tasks of `02b_tool_calling_at_the_desk.ipynb`. The setup repeats the
notebook's lookups and schemas, so this file runs on its own.

## Setup

In [1]:
from pathlib import Path
import json
import os
import sqlite3
from dotenv import dotenv_values

# The repository folder is the first folder, from here upwards, that holds models.env
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "models.env").exists())

# Models and prices come from models.env; a non-empty value in .env replaces one of them
settings = dotenv_values(ROOT / "models.env")
local = dotenv_values(ROOT / ".env")
settings.update({name: value for name, value in local.items() if name in settings and value})

# The key comes from the environment or .env, and is never printed
os.environ.setdefault("OPENAI_API_KEY", local.get("OPENAI_API_KEY") or "")
assert os.environ["OPENAI_API_KEY"], "Put OPENAI_API_KEY in .env at the repository folder (copy .env.example)."

MODEL = settings["MODEL_DEFAULT"]
EFFORT = settings["REASONING_EFFORT"] or None
print("model:", MODEL, "| reasoning effort:", EFFORT, "| key found: yes")

model: gpt-6-luna | reasoning effort: none | key found: yes


In [2]:
from openai import OpenAI, omit

client = OpenAI(timeout=60)

DB_PATH = ROOT / "data" / "chinook.db"
conn = sqlite3.connect(f"file:{DB_PATH}?mode=ro", uri=True, check_same_thread=False)

SYSTEM = "You help the support reps of a music shop. Answer in one or two sentences."


def find_customer(name: str) -> list[dict]:
    """Find a shop customer by full name. Returns a list of matches with their customer_id.

    Args:
        name: First and last name, for example 'Ana Silva'.
    """
    rows = conn.execute(
        "SELECT CustomerId, FirstName || ' ' || LastName, Country FROM customers "
        "WHERE FirstName || ' ' || LastName = ?", (name,)).fetchall()
    return [{"customer_id": r[0], "name": r[1], "country": r[2]} for r in rows]


def get_invoices(customer_id: int) -> dict:
    """Count a customer's invoices and add up their total in US dollars."""
    count, total = conn.execute(
        "SELECT COUNT(*), ROUND(SUM(Total), 2) FROM invoices WHERE CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "invoice_count": count, "total": total}


find_customer_tool = {
    "type": "function",
    "function": {
        "name": "find_customer",
        "description": "Find a shop customer by full name. Returns a list of matches with their customer_id.",
        "parameters": {
            "type": "object",
            "properties": {
                "name": {"type": "string", "description": "First and last name, for example 'Ana Silva'."}
            },
            "required": ["name"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}

get_invoices_tool = {
    "type": "function",
    "function": {
        "name": "get_invoices",
        "description": "Count a customer's invoices and add up their total in US dollars.",
        "parameters": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "integer", "description": "The customer's customer_id."}
            },
            "required": ["customer_id"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}

## Task 1: a third lookup

The function from the challenge, called by hand for customer 6. The query joins two tables: `customers` holds the
number of each customer's rep in `SupportRepId`, and `employees` holds the rep's name under that number.
`row[0] if row else None` returns `None` for a customer id that does not exist, where `row[0]` alone would fail.

In [3]:
def get_support_rep(customer_id: int) -> dict:
    """Name the support rep who looks after a customer."""
    row = conn.execute(
        "SELECT e.FirstName || ' ' || e.LastName FROM customers c "
        "JOIN employees e ON e.EmployeeId = c.SupportRepId WHERE c.CustomerId = ?", (customer_id,)).fetchone()
    return {"customer_id": customer_id, "support_rep": row[0] if row else None}


print(get_support_rep(6))

{'customer_id': 6, 'support_rep': 'Steve Johnson'}


Customer 6's rep is Steve Johnson. The schema follows `get_invoices_tool`: one required `integer`.

In [4]:
get_support_rep_tool = {
    "type": "function",
    "function": {
        "name": "get_support_rep",
        "description": "Name the support rep who looks after a customer.",
        "parameters": {
            "type": "object",
            "properties": {
                "customer_id": {"type": "integer", "description": "The customer's customer_id."}
            },
            "required": ["customer_id"],
            "additionalProperties": False,
        },
        "strict": True,
    },
}

TOOLS = [find_customer_tool, get_invoices_tool, get_support_rep_tool]

The round trip, with all three tools on offer.

In [5]:
messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "Who is the support rep of customer 6?"},
]

# The model asks
response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, reasoning_effort=EFFORT or omit)
reply = response.choices[0].message
call = reply.tool_calls[0]
print("the model asks for:", call.function.name, call.function.arguments)

# Our code runs the lookup
result = get_support_rep(**json.loads(call.function.arguments))
print("the lookup returns:", result)

# Our code sends the result back
messages.append(reply)
messages.append({"role": "tool", "tool_call_id": call.id, "content": json.dumps(result)})
response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, reasoning_effort=EFFORT or omit)
print("the model answers: ", response.choices[0].message.content)

the model asks for: get_support_rep {"customer_id":6}
the lookup returns: {'customer_id': 6, 'support_rep': 'Steve Johnson'}


the model answers:  Customer 6’s support representative is Steve Johnson.


Of the three tools the model asked for `get_support_rep`, with `{"customer_id":6}`, and the reply names Steve
Johnson: the name our hand call returned.

**Notice** that our code called `get_support_rep` directly, because we knew which tool this question needs. Code
that serves any question has to read `call.function.name` and run the function with that name.

## Task 2: an empty result

Part C's question again. When the model asks for `find_customer` we do not run it. We send back `"[]"`, the text of
an empty list.

In [6]:
messages = [
    {"role": "system", "content": SYSTEM},
    {"role": "user", "content": "Is Helena Holý one of our customers? Which country is she in?"},
]

response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, reasoning_effort=EFFORT or omit)
reply = response.choices[0].message
call = reply.tool_calls[0]
print("the model asks for:", call.function.name, call.function.arguments)

messages.append(reply)
messages.append({"role": "tool", "tool_call_id": call.id, "content": "[]"})

response = client.chat.completions.create(model=MODEL, messages=messages, tools=TOOLS, reasoning_effort=EFFORT or omit)
print("finish_reason:     ", response.choices[0].finish_reason)
print("the model answers: ", response.choices[0].message.content)

the model asks for: find_customer {"name":"Helena Holý"}


finish_reason:      stop
the model answers:  I couldn’t find a customer named Helena Holý in our records, so I can’t confirm her country.


In our run the model said it could not find a customer named Helena Holý in our records, so it could not confirm her country. `finish_reason` is `stop`: a finished answer, with no second try.

Helena Holý is customer 6; `find_customer("Helena Holý")` finds her. The model was told that the lookup found
nobody, and it answered from that. An empty result is a result: a lookup that fails quietly (a wrong table, a
name spelt another way, a filter that is too narrow) gives the same reply as a customer who does not exist.